# 04 - Evaluation

Test the two models from `03_training` on the test set,
and use Grad-CAM to see what the models look at. The cropped model is the final model (chosen in `03_training`).

## 1. Imports

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 2. Device and paths

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROCESSED_DIR = Path("..") / "data" / "processed"
FULL_DIR = PROCESSED_DIR / "full"
CROPPED_DIR = PROCESSED_DIR / "cropped"
MODEL_DIR = Path("..") / "outputs" / "models"

device

device(type='cuda')

## 3. Load the models

Same setup as in `03_training`: a ResNet18 with a last layer of 4 outputs.
Then the saved weights of the best epoch are loaded into it.
`eval()` puts the model in evaluation mode, because we only predict here, we don't train.

In [6]:
full_model = models.resnet18()
full_model.fc = nn.Linear(full_model.fc.in_features, 4)
full_model.load_state_dict(torch.load(MODEL_DIR / "resnet18_full.pt", map_location=device))
full_model = full_model.to(device).eval()

cropped_model = models.resnet18()
cropped_model.fc = nn.Linear(cropped_model.fc.in_features, 4)
cropped_model.load_state_dict(torch.load(MODEL_DIR / "resnet18_cropped.pt", map_location=device))
cropped_model = cropped_model.to(device).eval()

## 4. Test data

Same transform as the validation set in `03_training`, so no augmentation.
`shuffle=False` keeps the predictions in the same order as the labels.

In [7]:
weights = models.ResNet18_Weights.DEFAULT
MEAN = weights.transforms().mean
STD = weights.transforms().std

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_test_data = datasets.ImageFolder(FULL_DIR / "test", transform=eval_transform)
cropped_test_data = datasets.ImageFolder(CROPPED_DIR / "test", transform=eval_transform)

full_test_loader = DataLoader(full_test_data, batch_size=32, shuffle=False)
cropped_test_loader = DataLoader(cropped_test_data, batch_size=32, shuffle=False)

labels = np.array(cropped_test_data.targets)
classes = cropped_test_data.classes

len(full_test_data), len(cropped_test_data)

(1467, 1467)

## 5. Predictions

For every test image the model gives 4 scores. `softmax` turns them into probabilities that add up to 1,
and the class with the highest probability is the prediction. The probabilities are kept for the error analysis.

In [8]:
full_probs = np.zeros((len(full_test_data), 4))
start = 0
with torch.no_grad():
    for images, _ in full_test_loader:
        outputs = full_model(images.to(device))
        full_probs[start:start + len(images)] = torch.softmax(outputs, dim=1).cpu().numpy()
        start += len(images)

cropped_probs = np.zeros((len(cropped_test_data), 4))
start = 0
with torch.no_grad():
    for images, _ in cropped_test_loader:
        outputs = cropped_model(images.to(device))
        cropped_probs[start:start + len(images)] = torch.softmax(outputs, dim=1).cpu().numpy()
        start += len(images)

full_preds = full_probs.argmax(axis=1)
cropped_preds = cropped_probs.argmax(axis=1)

## 6. Scores

### 6.1 Accuracy

In [9]:
pd.Series({
    "full": (full_preds == labels).mean(),
    "cropped": (cropped_preds == labels).mean(),
}, name="test accuracy").round(4)

full       0.9250
cropped    0.9298
Name: test accuracy, dtype: float64

### 6.2 Per class

- **Precision:** of all images the model called this class, how many were right.
- **Recall:** of all images that really are this class, how many the model found. For tumors this is the one that matters most: a missed tumor is worse than a false alarm.

In [10]:
cropped_report = pd.DataFrame(classification_report(labels, cropped_preds, target_names=classes, output_dict=True)).T
full_report = pd.DataFrame(classification_report(labels, full_preds, target_names=classes, output_dict=True)).T

pd.concat([full_report.loc[classes, ["precision", "recall"]], cropped_report.loc[classes, ["precision", "recall"]]], axis=1, keys=["full", "cropped"]).round(3)

full          cropped       
           precision recall precision recall
glioma         0.994  0.850     0.976  0.861
meningioma     0.924  0.877     0.953  0.879
notumor        0.819  0.993     0.801  1.000
pituitary      0.964  0.992     0.990  0.992

### 6.3 Conclusions

- On the test set the cropped model gets 93.0% and the full model 92.5%. That's a lot lower than the 98.6% on validation.
- The big problem: notumor has a recall of 100%, but a precision of only 80%. The model calls too many images "notumor",
  and most of those are actually glioma or meningioma. So it misses tumors, which is the worst kind of mistake here.
- Pituitary is almost perfect (99%).
- Cropped is a bit better than full again, but the difference is small (7 images).
- The validation set came from the original training folder, the test set from the original test folder. The gap between them
  suggests the test images are different from what the model saw during training. Next step: look at the mistakes to find out why.